# Integrating 8-bit Mamba C Implementation with Python

In this notebook, we’ll load the 8-bit Mamba model implemented in C as a shared library, define Python wrapper functions, and run an example forward pass.

### Step 1: Load Required Libraries

In [1]:
import ctypes
import numpy as np


### Step 2: Load the Shared Library and Define the C Function Signatures

We load the compiled shared library (`libmamba_model.so`) and define the function signatures for `mamba_init` and `mamba_forward` using Python’s `ctypes` library.

In [12]:
# Load the shared library
mamba_lib = ctypes.CDLL('./8bit/libmamba_model.so')  # Adjust path if necessary

# Define argument and return types for the functions
mamba_lib.mamba_init.restype = None
mamba_lib.mamba_forward.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.int8, ndim=3, flags="C_CONTIGUOUS"),  # Input
    np.ctypeslib.ndpointer(dtype=np.int8, ndim=3, flags="C_CONTIGUOUS")   # Output
]
mamba_lib.mamba_forward.restype = None  # No return value


### Step 3: Create Python Wrapper Functions for Initialization and Forward Pass

Define wrapper functions to initialize the model parameters (`mamba_init`) and perform the forward pass (`mamba_forward`).

In [13]:
# Initialize the Mamba model parameters
def mamba_init():
    """Initialize the 8-bit Mamba model parameters."""
    mamba_lib.mamba_init()

# Perform the forward pass
def mamba_forward(input_array, output_shape):
    """
    Perform the forward pass.
    
    Parameters:
        input_array (np.ndarray): The input data of shape (BATCH_SIZE, SEQ_LENGTH, INPUT_DIM)
        output_shape (tuple): The shape of the output array.
    
    Returns:
        np.ndarray: The output data.
    """
    # Ensure the input is a contiguous array of type int8
    input_array = np.ascontiguousarray(input_array, dtype=np.int8)
    
    # Create an empty output array with the correct shape and type
    output_array = np.zeros(output_shape, dtype=np.int8)
    
    # Call the C function
    mamba_lib.mamba_forward(input_array, output_array)
    
    return output_array


### Step 4: Set Up Parameters and Test the Forward Pass

Initialize the model, create sample input data, and run the forward pass to ensure the integration works correctly.

In [15]:
# Define model parameters
BATCH_SIZE = 4
SEQ_LENGTH = 10
INPUT_DIM = 8
STATE_DIM = 16
OUTPUT_DIM = 5

# Initialize the Mamba model
mamba_init()

# Create a sample input array with random values
input_array = np.random.randint(-128, 127, (BATCH_SIZE, SEQ_LENGTH, INPUT_DIM), dtype=np.int8)

# Define the output shape
output_shape = (BATCH_SIZE, SEQ_LENGTH, OUTPUT_DIM)

# Perform the forward pass
output_array = mamba_forward(input_array, output_shape)

print("Output from 8-bit Mamba forward pass:", output_array)


Output from 8-bit Mamba forward pass: [[[ -35  -35  -35  -35    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0   36]
  [  36   36   36    0    0]
  [   0    0    0    0    0]
  [   0 -114 -114 -114 -114]
  [   0    0    0    0    0]
  [   0    0    0    0    0]]

 [[   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]]

 [[   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]]

 [[   0    0    0    0    0]
  [   0    0    0    0    0]
  [   0    0    0    0    0]